In [1]:
from typing import Optional
import tkinter as tk
import tkinter.font as tkFont
from nltk.sentiment.vader import SentimentIntensityAnalyzer

`from typing import Optional`

**Purpose:**  
Provide type-hinting tools for clearer function signatures.

**Behavior:**
- `Optional[T]` means “either `T` or `None`”.

**Effect:**  
Improves readability and static analysis without changing runtime behavior.

**Plain meaning:**  
Describe what kinds of values functions expect and return.

---

`import tkinter as tk`

**Purpose:**  
Import the GUI toolkit used to create windows and widgets.

**Behavior:**
- Provides the base objects for windows, layouts, and controls.

**Effect:**  
Allows the program to display a graphical interface.

**Plain meaning:**  
Create windows and on-screen controls.

---

`import tkinter.font as tkFont`

**Purpose:**  
Access font configuration tools for Tkinter widgets.

**Behavior:**
- Allows creation of custom fonts (family, size, weight).

**Effect:**  
Controls how text appears in the GUI.

**Plain meaning:**  
Choose how the text looks on screen.

---

`from nltk.sentiment.vader import SentimentIntensityAnalyzer`

**Purpose:**  
Import the VADER sentiment analyzer.

**Behavior:**
- Provides a ready-made model that scores text as positive, neutral, or negative.
- Returns a “compound” score between -1 and 1.

**Effect:**  
Enables automatic sentiment analysis of sentences.

**Plain meaning:**  
Measure how positive or negative a sentence is.


In [2]:
def __init__(
        self,
        title: str = "Sentiment Viewer",
        width: int = 80,
        height: int = 25,
        console_output: bool = False,
    ):
        self.console_output = console_output
        self.root = tk.Tk()
        self.root.title(title)

        self.custom_font = tkFont.Font(family="Courier", size=24, weight="bold")

        self.text_widget = tk.Text(
            self.root,
            font=self.custom_font,
            wrap="word",
            width=width,
            height=height,
            bg="#000000",
            fg="#FFFFFF",
            insertbackground="#FFFFFF",
        )
        self.scrollbar = tk.Scrollbar(self.root, command=self.text_widget.yview)
        self.text_widget.configure(yscrollcommand=self.scrollbar.set)

        self.text_widget.pack(side="left", fill="both", expand=True)
        self.scrollbar.pack(side="right", fill="y")
        self.analyzed_sentences = []

        self.sid = SentimentIntensityAnalyzer()

## `__init__(self, title: str = "Sentiment Viewer", width: int = 80, height: int = 25, console_output: bool = False)`

**Purpose:**  
Create the window and text area used to display sentiment-colored sentences.

**Behavior:**
- Stores whether extra printing to the console should happen (`console_output`).
- Creates a Tkinter window and sets its title.
- Creates a large monospace font for easy reading.
- Creates a scrolling text widget (black background, white text).
- Prepares an empty list to store analyzed sentences.
- Creates a VADER sentiment analyzer to score text.

**Effect:**  
After construction, the viewer is ready to display lines of text with sentiment-based colors.

**Plain meaning:**  
Set up a simple “text viewer app” that can show sentences and color them based on sentiment.


In [3]:
@staticmethod
def value_to_hex(val: float) -> str:
    if val == 0:
        return "#0066ff"  # light blue
    r = int(max(0, 255 * (1 - (val + 1) / 2)))
    g = int(max(0, 255 * ((val + 1) / 2)))
    b = 0
    return f"#{r:02X}{g:02X}{b:02X}"

## `value_to_hex(val: float) -> str`

**Purpose:**  
Convert a sentiment score into a color.

**Behavior:**
- If the score is exactly `0`, returns a fixed blue color.
- Otherwise computes red/green intensity based on the score value.
- Produces a hex color string like `#RRGGBB`.

**Effect:**  
Creates a consistent mapping from “negative/positive” to a visible color.

**Plain meaning:**  
Turn “how negative or positive” into a color the screen can show.


In [4]:
def add_line(self, sentence: str, score: Optional[float] = None) -> None:
        if score is not None:
            ss = {"compound": score}
            colour = self.value_to_hex(score)
        else:
            try:
                ss = self.sid.polarity_scores(sentence)
                colour = self.value_to_hex(ss["compound"])
            except Exception as e:
                ss = {"neg": 0.0, "neu": 0.0, "pos": 0.0, "compound": 0.0}
                colour = "#FFC0CB"
                sentence += f" [Error analyzing sentiment: {e}]"

        if self.console_output:
            print(sentence)
            for k in sorted(ss):
                print(f"{k}: {ss[k]}, ", end="")
            print(f"colour: {colour}\n")

        sentence = f"{sentence}  [compound: {ss['compound']}]"
        self.analyzed_sentences.append(sentence)

        self.text_widget.insert("end", sentence + "\n")

        line_start = f"{float(self.text_widget.index('end')) - 2} linestart"
        line_end = f"{float(self.text_widget.index('end')) - 1} lineend"
        tag_name = f"line{float(self.text_widget.index('end'))}"
        self.text_widget.tag_add(tag_name, line_start, line_end)
        self.text_widget.tag_config(tag_name, foreground=colour)

        self.text_widget.see("end")


## `add_line(self, sentence: str, score: Optional[float] = None) -> None`

**Purpose:**  
Analyze a sentence (or use a provided score), display it, and color it.

**Behavior:**
- If `score` is provided, uses it directly as the compound score.
- If no `score` is provided, runs VADER sentiment analysis on the sentence.
- If analysis fails, uses neutral values and marks the sentence with an error note.
- Optionally prints details to the console if `console_output` is enabled.
- Appends the compound score to the displayed sentence.
- Inserts the sentence into the GUI text widget.
- Applies a colored tag to the inserted line so the whole line changes color.
- Scrolls the view to the bottom so the newest line is visible.

**Effect:**  
Each call adds one new, color-coded line to the viewer and stores it for later export.

**Plain meaning:**  
Take a sentence, figure out its sentiment, show it on screen, and color it.


In [5]:
def run(self) -> None:
        self.root.mainloop()

## `run(self) -> None`

**Purpose:**  
Start the GUI so the window stays open and responds to user actions.

**Behavior:**
- Enters Tkinter’s main event loop.

**Effect:**  
The window becomes interactive and stays visible until closed.

**Plain meaning:**  
Open the viewer window and keep it running.